# Hackathón de Modelos Predictivos de Riesgo de Desplazamiento Interno por Desastres
### Notebook de entrega — equipo **AI COLOMBIA** · PNUD Colombia · 6 de octubre de 2026
Integrante: Juan Felipe Rodríguez Granados

Basado en el notebook base oficial: se conservan la carga de datos, `validar_entrega` y el formato del archivo de
salida (`entrega_AICOLOMBIA.csv`). Se reemplaza el modelo simple por el modelo del equipo, **E10S_LL**:

| Componente | Decisión |
|---|---|
| Historial (sin fuga) | Para cada año, solo los años anteriores . Tasas suavizadas municipio → departamento → nacional, por mes del año, magnitud típica, mezcla de amenazas y tasa de los últimos 12 meses |
| Contexto espacial | Cartografía pública de la UPRA (derivada del MGN del DANE): latitud, longitud, área, población (densidad × área) y tasa histórica de los 5 municipios más cercanos |
| Lluvia | CHIRPS v2.0 mensual, **solo 2018-01 → 2025-12** (los meses del reto): nivel, acumulado de 3 meses, anomalías frente a lo normal del municipio en ese mes del año (climatología con 2018-01 → 2022-09) e interacción con el historial |
| Modelos | 4 LightGBM: clasificador (100 árboles, 15 hojas, mínimo 200 filas por hoja) → `prob_evento`; regresión sobre log1p(personas) → estimado; cuantiles 0.1 y 0.9 → intervalo (factor de q90 que minimiza el Winkler) |
| Validación | 3 cortes temporales (F1, F2, F3) con las métricas oficiales: **25.56 puntos** frente a 21.98 sin lluvia y 20.06 de nuestra primera versión |

Secciones: 1. Cargar · 2. Explorar y tratar · 3. Historial, cartografía y lluvia · 4. Variables y modelo ·
5. Validación temporal · 6. Entrenar con todo y generar la entrega · 7. Otros entregables y declaración de IA.

**Para reproducir:** *Entorno de ejecución → Ejecutar todo*. Descarga los datos del reto, la cartografía (UPRA) y la
lluvia (CHIRPS, unos 100–150 MB en lecturas parciales; instala `rasterio` si falta). Tarda unos minutos.

In [1]:
INCLUIR_LLUVIA = True   # decisión del equipo: modelo E10S_LL. Con False se entrega E10S (sin lluvia)

## 1. Cargar los datos
Enlace de Google Drive compartido por los organizadores (`URL_DATOS`). Si prefieren, suban el zip a mano con el ícono de
carpeta de la izquierda y dejen `URL_DATOS` vacío.

In [2]:
import os, zipfile
import numpy as np
import pandas as pd

URL_DATOS = "PEGUE_AQUI_EL_ENLACE_DE_LOS_DATOS"  # enlace de Google Drive al archivo hackathon_datos.zip (se comparte el día del evento)
DATA_DIR = os.environ.get("HACKATHON_DATA", "datos")

if not os.path.exists(os.path.join(DATA_DIR, "entrenamiento.csv")):
    if URL_DATOS:
        import gdown
        import re
        ident = re.search(r"/d/([\w-]+)", URL_DATOS)   # enlace directo: funciona con cualquier versión de gdown
        url = f"https://drive.google.com/uc?id={ident.group(1)}" if ident else URL_DATOS
        gdown.download(url, "hackathon_datos.zip", quiet=False)
    with zipfile.ZipFile("hackathon_datos.zip") as z:
        z.extractall(DATA_DIR)

leer = lambda f: pd.read_csv(os.path.join(DATA_DIR, f), dtype={"DIVIPOLA": str}, parse_dates=["fecha"])
train = leer("entrenamiento.csv")
test_pub = leer("prueba_equipos.csv")     # periodo de la tabla de posiciones en vivo
test_priv = leer("prueba_oculta.csv")     # periodo de la nota oficial
dic = pd.read_csv(os.path.join(DATA_DIR, "diccionario_datos.csv"))
print(train.shape, test_pub.shape, test_priv.shape)

Downloading...
From: https://drive.google.com/uc?id=<ID_OCULTO>
To: /content/hackathon_datos.zip
100%|██████████| 959k/959k [00:00<00:00, 3.43MB/s]


(62814, 25) (20938, 13) (22040, 13)


## 2. Explorar y tratar
La variable objetivo solo está en `entrenamiento.csv`. Los archivos de prueba traen identificadores y covariables.

In [3]:
pd.set_option("display.max_colwidth", 120)
display(dic[["columna", "archivos", "grupo", "descripcion"]])
print("Prevalencia de tiene_evento:", round(train.tiene_evento.mean(), 3))
print("Meses con evento por año:")
display(train.groupby("anio").tiene_evento.mean().round(3).to_frame())
print("Personas desplazadas cuando hay evento:")
display(train.loc[train.tiene_evento == 1, "personas_desplazadas"].describe().round(1).to_frame())

,columna,archivos,grupo,descripcion
0,DIVIPOLA,todos,Identificador,Código DANE del municipio. Léalo como texto: los ceros a la izquierda son significativos (ej. 05001).
1,fecha,todos,Identificador,Primer día del mes de la observación.
2,anio,todos,Identificador,Año de la fecha.
3,mes,todos,Identificador,Mes calendario de la fecha.
4,tiene_evento,entrenamiento,Variable objetivo,"1 si el municipio tuvo al menos un evento climático con afectación ese mes: personas afectadas > 0, viviendas destru..."
5,personas_desplazadas,entrenamiento,Variable objetivo,Suma de personas reportadas como afectadas por los eventos climáticos del mes. Vale 0 cuando tiene_evento = 0. Se pr...
6,n_eventos,entrenamiento,Historial,Número de eventos climáticos con afectación reportados en el municipio ese mes.
7,familias_desplazadas,entrenamiento,Historial,Suma de familias reportadas como afectadas por los eventos del mes.
8,viv_destruidas,entrenamiento,Historial,Suma de viviendas destruidas por los eventos del mes.
9,viv_averiadas,entrenamiento,Historial,Suma de viviendas averiadas por los eventos del mes.


Prevalencia de tiene_evento: 0.079
Meses con evento por año:


,tiene_evento
anio,
2018,0.060
2019,0.057
2020,0.072
2021,0.100
2022,0.115


Personas desplazadas cuando hay evento:


,personas_desplazadas
count,4977.0
mean,439.5
std,2621.4
min,0.0
25%,8.0
50%,36.0
75%,170.0
max,150699.0


### Calidad y tratamiento
Se revisan las filas esperadas, los nulos y los valores fuera de rango. Tratamiento: duplicados, objetivo según la
definición oficial, valores imposibles → nulo e imputación por temporalidad con banderas `_imputado`.

In [4]:
ESPERADAS = {"entrenamiento": 62_814, "prueba_equipos": 20_938, "prueba_oculta": 22_040}
for nombre, df in {"entrenamiento": train, "prueba_equipos": test_pub, "prueba_oculta": test_priv}.items():
    print(f"{nombre:15s} filas={len(df):,} (esperadas {ESPERADAS[nombre]:,}) · municipios={df.DIVIPOLA.nunique()} · "
          f"llaves repetidas={df.duplicated(['DIVIPOLA', 'fecha']).sum()} · celdas nulas={int(df.isna().sum().sum())}")
nulos = pd.concat({"entrenamiento": train.isna().sum(), "prueba_equipos": test_pub.isna().sum(),
                   "prueba_oculta": test_priv.isna().sum()}, axis=1)
display(nulos[nulos.sum(axis=1) > 0])
fuera = pd.concat([test_pub, test_priv]).ONI.gt(train.ONI.max()).mean()
print(f"ONI de prueba por encima del máximo del entrenamiento: {fuera:.1%} de las filas (los árboles no extrapolan)")

entrenamiento   filas=62,814 (esperadas 62,814) · municipios=1102 · llaves repetidas=0 · celdas nulas=162
prueba_equipos  filas=20,938 (esperadas 20,938) · municipios=1102 · llaves repetidas=0 · celdas nulas=38
prueba_oculta   filas=22,040 (esperadas 22,040) · municipios=1102 · llaves repetidas=0 · celdas nulas=40


,entrenamiento,prueba_equipos,prueba_oculta
densidad_poblacional,12,0.0,0.0
ingresos_tributarios,36,0.0,0.0
inversion_gestion_riesgo,57,19.0,20.0
nbi_pct,57,19.0,20.0


ONI de prueba por encima del máximo del entrenamiento: 20.5% de las filas (los árboles no extrapolan)


In [5]:
COV_ESTATICAS = ["nbi_pct", "elevacion_media", "pendiente_media", "pct_pendiente_alta"]
COV_ANUALES = ["densidad_poblacional", "ingresos_tributarios", "inversion_gestion_riesgo"]
RANGOS = {"densidad_poblacional": (0, None), "ingresos_tributarios": (0, None), "inversion_gestion_riesgo": (0, None),
          "nbi_pct": (0, 100), "elevacion_media": (-50, 6000), "pendiente_media": (0, 90),
          "pct_pendiente_alta": (0, 100), "ONI": (-3.5, 3.5)}
HISTORIAL = ["n_eventos", "familias_desplazadas", "viv_destruidas", "viv_averiadas", "n_movimiento_masa",
             "n_inundacion", "n_vendaval", "n_creciente_subita", "n_avenida_torrencial", "n_otros"]
OBJETIVO = ["tiene_evento", "personas_desplazadas"]


def deduplicar(df):
    df = df.drop_duplicates()
    df = (df.assign(_nulos=df.isna().sum(axis=1)).sort_values(["DIVIPOLA", "fecha", "_nulos"])
            .drop_duplicates(["DIVIPOLA", "fecha"]).drop(columns="_nulos"))
    return df.sort_values(["DIVIPOLA", "fecha"]).reset_index(drop=True)


def preparar(train, test_pub, test_priv):
    partes = {}
    for k, df in {"train": train, "pub": test_pub, "priv": test_priv}.items():
        df = df.copy()
        df["DIVIPOLA"] = df["DIVIPOLA"].astype(str).str.strip().str.zfill(5)
        df["anio"], df["mes"] = df["fecha"].dt.year, df["fecha"].dt.month
        n0 = len(df)
        partes[k] = deduplicar(df)
        if len(partes[k]) != n0:
            print(f"{k}: {n0 - len(partes[k])} filas duplicadas eliminadas")

    tr = partes["train"]
    tr["personas_desplazadas"] = tr["personas_desplazadas"].clip(lower=0)
    tr[HISTORIAL] = tr[HISTORIAL].clip(lower=0).fillna(0)
    viv = tr[["viv_destruidas", "viv_averiadas"]].sum(axis=1)
    afectacion = (tr["personas_desplazadas"].fillna(0) > 0) | (viv > 0)
    corregidas = int(((tr["tiene_evento"] != 1) & afectacion).sum())
    tr["tiene_evento"] = np.where(afectacion, 1, tr["tiene_evento"].fillna(0)).astype(int)
    tr["personas_desplazadas"] = tr["personas_desplazadas"].fillna(0)
    print(f"Objetivo corregido según la definición oficial: {corregidas} filas")

    panel = pd.concat([d.assign(_parte=k, _i=np.arange(len(d))) for k, d in partes.items()], ignore_index=True)
    for c, (lo, hi) in RANGOS.items():
        malo = pd.Series(False, index=panel.index)
        if lo is not None:
            malo |= panel[c] < lo
        if hi is not None:
            malo |= panel[c] > hi
        if malo.any():
            print(f"{c}: {int(malo.sum())} valores imposibles -> nulo")
        panel.loc[malo, c] = np.nan
    faltaba = {c: panel[c].isna() for c in RANGOS if panel[c].isna().any()}

    panel["ONI"] = panel["ONI"].fillna(panel.groupby("fecha")["ONI"].transform("median"))
    for c in COV_ESTATICAS:
        panel[c] = panel[c].fillna(panel.groupby("DIVIPOLA")[c].transform("median"))
    for c in COV_ANUALES:   # el panel está en orden cronológico dentro de cada municipio
        panel[c] = panel[c].fillna(panel.groupby(["DIVIPOLA", "anio"])[c].transform("median"))
        panel[c] = panel.groupby("DIVIPOLA")[c].transform(lambda s: s.ffill().bfill())
    dep = panel["DIVIPOLA"].str[:2]
    for c in COV_ESTATICAS + COV_ANUALES:
        grupos = [dep, panel["anio"]] if c in COV_ANUALES else [dep]
        panel[c] = panel[c].fillna(panel.groupby(grupos)[c].transform("median")).fillna(panel[c].median())
    for c, mascara in faltaba.items():
        panel[c + "_imputado"] = mascara.astype(int)
        print(f"{c}: {int(mascara.sum()):,} valores imputados")
    panel["fase_enso"] = np.select([panel.ONI >= 0.5, panel.ONI <= -0.5], ["El Nino", "La Nina"], "Neutral")

    salida = {k: panel[panel._parte == k].sort_values("_i").drop(columns=["_parte", "_i"]).reset_index(drop=True)
              for k in partes}
    salida["train"]["tiene_evento"] = salida["train"]["tiene_evento"].astype(int)
    for k in ("pub", "priv"):
        salida[k] = salida[k].drop(columns=OBJETIVO + HISTORIAL, errors="ignore")
    return salida["train"], salida["pub"], salida["priv"]


train, test_pub, test_priv = preparar(train, test_pub, test_priv)
print(train.shape, test_pub.shape, test_priv.shape)

Objetivo corregido según la definición oficial: 0 filas
densidad_poblacional: 12 valores imputados
ingresos_tributarios: 36 valores imputados
inversion_gestion_riesgo: 96 valores imputados
nbi_pct: 96 valores imputados
(62814, 29) (20938, 17) (22040, 17)


## 3. Historial de cada municipio (sin fuga de información)
* **Entrenamiento:** las filas de cada año ven solo los años **anteriores** (2018 es de calentamiento).
* **Prueba:** historial calculado con **todo** el entrenamiento, congelado en 2022-09.
* Las tasas se suavizan hacia el departamento y el promedio nacional (pocos eventos por municipio = mucho ruido).

In [6]:
K = {"dep": 120, "nm": 1102, "dm": 60, "mun": 24, "mm": 3, "mag": 3}
HIST_COLS = ["hist_tasa_mun", "hist_tasa_mes", "hist_tasa_dep", "hist_tasa_dep_mes", "hist_tasa_nac_mes",
             "hist_mag_mun", "hist_esperado_log", "hist_pct_inund", "hist_pct_masa", "hist_reciente_12"]


def _conteos(hist, claves, p):
    return (hist.groupby(claves).agg(**{f"e_{p}": ("tiene_evento", "sum"), f"n_{p}": ("tiene_evento", "size"),
                                         f"lp_{p}": ("logp", "sum")}).reset_index())


def historial_congelado(hist, filas):
    """Historial calculado con todo `hist` (que es pasado respecto a `filas`), aplicado a `filas`."""
    hist = hist.assign(logp=np.log1p(hist["personas_desplazadas"]), dep=hist["DIVIPOLA"].str[:2])
    f = filas.assign(dep=filas["DIVIPOLA"].str[:2])
    for claves, p in [(["DIVIPOLA"], "mun"), (["DIVIPOLA", "mes"], "mm"), (["dep"], "dep"),
                      (["dep", "mes"], "dm"), (["mes"], "nm")]:
        f = f.merge(_conteos(hist, claves, p), on=claves, how="left")
    c = f.fillna({col: 0 for col in f.columns if col[:2] in ("e_", "n_") or col.startswith("lp_")})
    tasa_nac = hist["tiene_evento"].mean()
    tasa_nm = (c.e_nm + K["nm"] * tasa_nac) / (c.n_nm + K["nm"])
    tasa_dep = (c.e_dep + K["dep"] * tasa_nac) / (c.n_dep + K["dep"])
    tasa_dm = (c.e_dm + K["dm"] * tasa_dep * tasa_nm / tasa_nac) / (c.n_dm + K["dm"])
    tasa_mun = (c.e_mun + K["mun"] * tasa_dep) / (c.n_mun + K["mun"])
    tasa_mm = (c.e_mm + K["mm"] * tasa_mun * tasa_dm / tasa_dep) / (c.n_mm + K["mm"])
    mag_nac = hist["logp"].sum() / max(hist["tiene_evento"].sum(), 1)
    mag_dep = (c.lp_dep + 5 * mag_nac) / (c.e_dep + 5)
    mag_mun = (c.lp_mun + K["mag"] * mag_dep) / (c.e_mun + K["mag"])

    tipos = hist.groupby("DIVIPOLA")[["n_inundacion", "n_movimiento_masa", "n_eventos"]].sum()
    reciente = hist[hist["fecha"] > hist["fecha"].max() - pd.DateOffset(months=12)].groupby("DIVIPOLA")["tiene_evento"].mean()
    salida = filas.copy()
    salida["hist_tasa_mun"] = tasa_mun.to_numpy()
    salida["hist_tasa_mes"] = tasa_mm.to_numpy()
    salida["hist_tasa_dep"] = tasa_dep.to_numpy()
    salida["hist_tasa_dep_mes"] = tasa_dm.to_numpy()
    salida["hist_tasa_nac_mes"] = tasa_nm.to_numpy()
    salida["hist_mag_mun"] = mag_mun.to_numpy()
    salida["hist_esperado_log"] = (tasa_mm * mag_mun).to_numpy()   # E[log1p(personas)] según la historia
    salida["hist_pct_inund"] = filas["DIVIPOLA"].map((tipos.n_inundacion + 1) / (tipos.n_eventos + 3)).fillna(1 / 3).to_numpy()
    salida["hist_pct_masa"] = filas["DIVIPOLA"].map((tipos.n_movimiento_masa + 1) / (tipos.n_eventos + 3)).fillna(1 / 3).to_numpy()
    salida["hist_reciente_12"] = filas["DIVIPOLA"].map(reciente).fillna(tasa_nac).to_numpy()
    return salida


def historial_entrenamiento(hist):
    """Filas de entrenamiento: para cada año Y (desde el segundo), historial con los años anteriores a Y."""
    anios = sorted(hist["anio"].unique())
    return pd.concat([historial_congelado(hist[hist.anio < a], hist[hist.anio == a]) for a in anios[1:]],
                     ignore_index=True)

### Cartografía municipal (fuente pública, declarada)
Capa "municipios_generalizada" de la UPRA, derivada del Marco Geoestadístico Nacional del DANE. Se descarga del
servicio público (o se lee de la caché) y se valida antes de usarla.

In [7]:
import json
import math
import urllib.parse
import urllib.request

CARTO_SERVICIO = ("https://geoservicios.upra.gov.co/arcgis/rest/services/referencia_geografica/"
                  "municipios_generalizada/MapServer/0/query")
CARTO_GEOJSON = "cartografia_municipios.geojson"   # caché local, se crea en la primera corrida
CARTO_CSV = "geografia.csv"                        # alternativa sin internet
KM_POR_GRADO_LAT, KM_POR_GRADO_LON = 110.574, 111.320


def descargar_cartografia(tolerancia=0.002):
    parametros = {"where": "1=1", "outFields": "cod_dane_mpio,municipio,departamen", "outSR": 4326,
                  "maxAllowableOffset": tolerancia, "geometryPrecision": 4, "resultRecordCount": 2000, "f": "geojson"}
    with urllib.request.urlopen(f"{CARTO_SERVICIO}?{urllib.parse.urlencode(parametros)}", timeout=120) as r:
        return json.loads(r.read().decode("utf-8"))


def centroide_y_area(geometria):
    """Centroide del polígono exterior más grande y área en km² (proyección local equirectangular)."""
    poligonos = [geometria["coordinates"]] if geometria["type"] == "Polygon" else geometria["coordinates"]
    area_km2, mejor, mejor_area = 0.0, None, -1.0
    for poligono in poligonos:
        lat_media = sum(y for _, y in poligono[0]) / len(poligono[0])
        escala = KM_POR_GRADO_LAT * KM_POR_GRADO_LON * math.cos(math.radians(lat_media))
        for k, anillo in enumerate(poligono):
            a = cx = cy = 0.0
            for (x0, y0), (x1, y1) in zip(anillo, anillo[1:] + anillo[:1]):
                cruz = x0 * y1 - x1 * y0
                a, cx, cy = a + cruz, cx + (x0 + x1) * cruz, cy + (y0 + y1) * cruz
            a /= 2
            area_km2 += (abs(a) if k == 0 else -abs(a)) * escala
            if k == 0 and a != 0 and abs(a) > mejor_area:
                mejor_area, mejor = abs(a), (cx / (6 * a), cy / (6 * a))
    return mejor, area_km2


def tabla_desde_geojson(geojson):
    filas = []
    for f in geojson["features"]:
        if f.get("geometry"):
            (lon, lat), area = centroide_y_area(f["geometry"])
            filas.append({"DIVIPOLA": str(f["properties"]["cod_dane_mpio"]).zfill(5),
                          "lat": round(lat, 4), "lon": round(lon, 4), "area_km2": round(area, 1)})
    return pd.DataFrame(filas).drop_duplicates("DIVIPOLA").reset_index(drop=True)


def cargar_cartografia():
    geojson, origen = None, None
    if os.path.exists(CARTO_GEOJSON):
        with open(CARTO_GEOJSON, encoding="utf-8") as f:
            geojson, origen = json.load(f), f"caché local ({CARTO_GEOJSON})"
    else:
        try:
            geojson, origen = descargar_cartografia(), "servicio público de la UPRA (MGN del DANE)"
            with open(CARTO_GEOJSON, "w", encoding="utf-8") as f:
                json.dump(geojson, f)
        except Exception as error:
            print(f"⚠ No se pudo descargar la cartografía: {error}")
    if geojson is not None:
        return tabla_desde_geojson(geojson), geojson, origen
    if os.path.exists(CARTO_CSV):
        tabla = pd.read_csv(CARTO_CSV, dtype={"DIVIPOLA": str})[["DIVIPOLA", "lat", "lon", "area_km2"]]
        return tabla, None, f"archivo subido ({CARTO_CSV}; sin polígonos: no hay vecindad por frontera)"
    return None, None, "no disponible"


def validar_cartografia(tabla, datos):
    t = tabla.set_index("DIVIPOLA")
    codigos = set(datos["DIVIPOLA"])
    faltan = sorted(codigos - set(tabla["DIVIPOLA"]))
    dens = datos.loc[(datos["DIVIPOLA"] == "11001") & (datos["anio"] == datos["anio"].max()), "densidad_poblacional"]
    pob_bogota = float(dens.iloc[0] * t["area_km2"].get("11001", np.nan)) if len(dens) else float("nan")
    bloqueantes = [
        ("al menos 1,100 municipios", len(tabla) >= 1100),
        ("códigos DIVIPOLA únicos y de 5 dígitos",
         tabla["DIVIPOLA"].is_unique and tabla["DIVIPOLA"].str.fullmatch(r"\d{5}").all()),
        ("centroides dentro de Colombia",
         tabla["lat"].between(-4.5, 13.6).all() and tabla["lon"].between(-82.0, -66.8).all()),
        ("áreas positivas", (tabla["area_km2"] > 0).all()),
        ("área de Bogotá ≈ 1,636 km² (±5 %)", abs(t["area_km2"].get("11001", 0) / 1636 - 1) < 0.05),
    ]
    informativas = [
        (f"población de Bogotá = densidad × área ≈ 7–8 millones (da {pob_bogota / 1e6:.1f} M)", 5e6 < pob_bogota < 1e7),
        (f"cubre los {len(codigos):,} municipios del reto (faltan {len(faltan)}: {faltan[:5]})", not faltan),
    ]
    for texto, ok in bloqueantes + informativas:
        print(("✅ " if ok else "❌ ") + texto)
    return all(ok for _, ok in bloqueantes)


GEO_TABLA, GEO_JSON, origen = cargar_cartografia()
USAR_ESPACIAL = GEO_TABLA is not None and validar_cartografia(GEO_TABLA, train)
print(f"Cartografía: {origen} → variantes espaciales {'ACTIVADAS' if USAR_ESPACIAL else 'DESACTIVADAS'}")

✅ al menos 1,100 municipios
✅ códigos DIVIPOLA únicos y de 5 dígitos
✅ centroides dentro de Colombia
✅ áreas positivas
✅ área de Bogotá ≈ 1,636 km² (±5 %)
✅ población de Bogotá = densidad × área ≈ 7–8 millones (da 7.8 M)
✅ cubre los 1,102 municipios del reto (faltan 0: [])
Cartografía: servicio público de la UPRA (MGN del DANE) → variantes espaciales ACTIVADAS


### Vecindad
Tasa histórica de los 5 municipios con el centroide más cercano, con la **misma historia que la fila** (sin fuga).

In [8]:
def pares_knn(tabla, k, ponderado=False):
    lat, lon = np.radians(tabla["lat"].to_numpy()), np.radians(tabla["lon"].to_numpy())
    a = (np.sin((lat[:, None] - lat[None, :]) / 2) ** 2
         + np.cos(lat)[:, None] * np.cos(lat)[None, :] * np.sin((lon[:, None] - lon[None, :]) / 2) ** 2)
    distancia = 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    np.fill_diagonal(distancia, np.inf)
    cercanos = np.argsort(distancia, axis=1)[:, :k]
    km = np.take_along_axis(distancia, cercanos, axis=1) * 6371
    peso = 1 / (km + 1) if ponderado else np.ones_like(km)
    codigos = tabla["DIVIPOLA"].to_numpy()
    return pd.DataFrame({"DIVIPOLA": np.repeat(codigos, k), "vecino": codigos[cercanos.ravel()], "peso": peso.ravel()})


VECINDADES = {}
if USAR_ESPACIAL:
    VECINDADES = {"k5": pares_knn(GEO_TABLA, 5)}   # los 5 municipios más cercanos
    for nombre, pares in VECINDADES.items():
        print(f"vecindad {nombre:8s}: {pares.groupby('DIVIPOLA').size().mean():.1f} vecinos en promedio")
    train, test_pub, test_priv = (d.merge(GEO_TABLA, on="DIVIPOLA", how="left") for d in (train, test_pub, test_priv))

VEC_COLS = {"hist_tasa_mun": "tasa", "hist_tasa_mes": "tasa_mes", "hist_mag_mun": "mag", "hist_reciente_12": "reciente"}
MESES = pd.DataFrame({"mes": range(1, 13)})


def agregar_vecindad(hist, filas):
    if not USAR_ESPACIAL:
        return filas
    grilla = GEO_TABLA[["DIVIPOLA"]].merge(MESES, how="cross")
    t = historial_congelado(hist, grilla)[["DIVIPOLA", "mes"] + list(VEC_COLS)].rename(columns={"DIVIPOLA": "vecino"})
    salida = filas
    for nombre, pares in VECINDADES.items():
        v = pares.merge(t, on="vecino")
        for c in VEC_COLS:
            v[c] = v[c] * v["peso"]
        g = v.groupby(["DIVIPOLA", "mes"])[list(VEC_COLS) + ["peso"]].sum()
        g = g[list(VEC_COLS)].div(g["peso"], axis=0)
        g.columns = [f"vec_{nombre}_{VEC_COLS[c]}" for c in VEC_COLS]
        salida = salida.merge(g.reset_index(), on=["DIVIPOLA", "mes"], how="left")
    # Suavizado espacial: tasa del municipio encogida hacia la de sus 5 vecinos (en lugar del departamento)
    conteo = hist.groupby("DIVIPOLA")["tiene_evento"].agg(["sum", "size"])
    e = salida["DIVIPOLA"].map(conteo["sum"]).fillna(0).to_numpy()
    n = salida["DIVIPOLA"].map(conteo["size"]).fillna(0).to_numpy()
    previa = salida["vec_k5_tasa"].fillna(hist["tiene_evento"].mean()).to_numpy()
    salida["hist_tasa_mun_espacial"] = (e + K["mun"] * previa) / (n + K["mun"])
    return salida


def historial_completo(hist, filas):
    return agregar_vecindad(hist, historial_congelado(hist, filas))


def historial_entrenamiento_completo(hist):
    anios = sorted(hist["anio"].unique())
    return pd.concat([historial_completo(hist[hist.anio < a], hist[hist.anio == a]) for a in anios[1:]],
                     ignore_index=True)

vecindad k5      : 5.0 vecinos en promedio


### Lluvia mensual por municipio (CHIRPS, fuente abierta y global)
CHIRPS v2.0 (Climate Hazards Center, UC Santa Barbara; Funk et al., 2015) estima la lluvia con satélite y estaciones
en una grilla de ~5 km. Se lee **solo la ventana de Colombia y solo los 96 meses del reto** (2018-01 → 2025-12) y se
promedian los pixeles dentro de cada municipio. Transformaciones:

* `lluvia_log`, `lluvia_3m_log`: log(1 + mm) del mes y del acumulado de 3 meses (suelo saturado).
* `lluvia_anom`, `lluvia_3m_anom`: diferencia frente al promedio del **mismo municipio y mismo mes del año**,
  calculado solo con 2018-01 → 2022-09. ¿Llovió más o menos de lo normal para ese lugar y esa época?
* `lluvia_anom_rezago1`: anomalía del mes anterior. `lluvia_anom_x_tasa`: anomalía × tasa histórica del mes.

En los meses de prueba es la lluvia ya observada de esos meses, como el ONI que traen los archivos. No contiene
información de emergencias.

In [9]:
import subprocess
import sys

LLUVIA_CSV = "lluvia_chirps_mensual.csv"
LLUVIA_CANDIDATAS = [LLUVIA_CSV, os.path.join("..", "data", "externos", "procesados", LLUVIA_CSV),
                     "/content/drive/MyDrive/Documentos Juan Felipe/Hackaton_PENUD/data/externos/procesados/" + LLUVIA_CSV]
CHIRPS_URL = "https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_monthly/cogs/chirps-v2.0.{a}.{m:02d}.cog"
MESES_RETO = pd.date_range("2018-01-01", "2025-12-01", freq="MS")   # solo los meses del reto
CLIMA_FIN = pd.Timestamp("2022-09-01")                               # climatología: solo el periodo de entrenamiento


def descargar_lluvia(geojson, hilos=8):
    """Lee de CHIRPS solo la ventana de Colombia de cada mes del reto y promedia los pixeles de cada municipio."""
    try:
        import rasterio
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio"], check=True)
        import rasterio
    from concurrent.futures import ThreadPoolExecutor

    import shapely
    from rasterio.windows import Window
    from shapely.geometry import shape

    os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")
    feats = [f for f in geojson["features"] if f.get("geometry")]
    codigos = [str(f["properties"]["cod_dane_mpio"]).zfill(5) for f in feats]
    geoms = [shape(f["geometry"]) for f in feats]
    b = np.array([g.bounds for g in geoms])
    oeste, sur, este, norte = b[:, 0].min() - 0.1, b[:, 1].min() - 0.1, b[:, 2].max() + 0.1, b[:, 3].max() + 0.1

    def leer(fecha):
        with rasterio.open("/vsicurl/" + CHIRPS_URL.format(a=fecha.year, m=fecha.month)) as src:
            f0, c0 = src.index(oeste, norte)
            f1, c1 = src.index(este, sur)
            ventana = Window(c0, f0, c1 - c0 + 1, f1 - f0 + 1)
            matriz = src.read(1, window=ventana).astype("float64")
            if src.nodata is not None:
                matriz[matriz == src.nodata] = np.nan
            matriz[matriz < 0] = np.nan
            return matriz, src.window_transform(ventana)

    with ThreadPoolExecutor(hilos) as ex:
        res = list(ex.map(leer, MESES_RETO))
    pila, tr = np.stack([r[0] for r in res]), res[0][1]
    _, alto, ancho = pila.shape
    validos = ~np.isnan(pila).any(axis=0)
    fv, cv = np.nonzero(validos)
    xs = tr.c + (np.arange(ancho) + 0.5) * tr.a
    ys = tr.f + (np.arange(alto) + 0.5) * tr.e
    plana = pila.reshape(len(MESES_RETO), -1)
    columnas = []
    for g in geoms:
        minx, miny, maxx, maxy = g.bounds
        cc, ff = np.meshgrid(np.nonzero((xs >= minx) & (xs <= maxx))[0], np.nonzero((ys >= miny) & (ys <= maxy))[0])
        sel = np.array([], dtype=int)
        if cc.size:
            dentro = shapely.contains_xy(g, xs[cc], ys[ff])
            sel = ff[dentro] * ancho + cc[dentro]
            sel = sel[validos.ravel()[sel]]
        if len(sel) == 0:   # municipio pequeño: el pixel válido más cercano a su punto representativo
            p = g.representative_point()
            k = int(np.argmin((xs[cv] - p.x) ** 2 + (ys[fv] - p.y) ** 2))
            sel = np.array([fv[k] * ancho + cv[k]])
        columnas.append(plana[:, sel].mean(axis=1))
    return pd.DataFrame({"DIVIPOLA": np.tile(codigos, len(MESES_RETO)),
                         "fecha": np.repeat(MESES_RETO, len(codigos)),
                         "lluvia_mm": np.round(np.column_stack(columnas).ravel(), 2)})


def cargar_lluvia():
    for ruta in LLUVIA_CANDIDATAS:
        if os.path.exists(ruta):
            t = pd.read_csv(ruta, dtype={"DIVIPOLA": str}, parse_dates=["fecha"])
            return t.assign(DIVIPOLA=t["DIVIPOLA"].str.zfill(5))[["DIVIPOLA", "fecha", "lluvia_mm"]], f"archivo ({ruta})"
    if GEO_JSON is None:
        return None, "no disponible (no hay polígonos para calcularla)"
    try:
        t = descargar_lluvia(GEO_JSON)
        t.to_csv(LLUVIA_CSV, index=False)
        return t, "descargada de CHIRPS (UC Santa Barbara)"
    except Exception as error:
        print(f"⚠ No se pudo descargar la lluvia: {error}")
        return None, "no disponible"


def validar_lluvia(t, llaves):
    cruce = llaves.merge(t, on=["DIVIPOLA", "fecha"], how="left")
    pruebas = [
        ("solo meses del reto (2018-01 → 2025-12)", t.fecha.min() >= MESES_RETO[0] and t.fecha.max() <= MESES_RETO[-1]),
        ("sin llaves duplicadas", not t.duplicated(["DIVIPOLA", "fecha"]).any()),
        ("lluvia entre 0 y 3,000 mm/mes", bool(t.lluvia_mm.between(0, 3000).all())),
        ("96 meses por municipio (para los acumulados y rezagos)", bool((t.groupby("DIVIPOLA").size() == 96).all())),
        (f"cubre todas las filas municipio-mes del reto (faltan {int(cruce.lluvia_mm.isna().sum())})",
         bool(cruce.lluvia_mm.notna().all())),
    ]
    for texto, ok in pruebas:
        print(("✅ " if ok else "❌ ") + texto)
    return all(ok for _, ok in pruebas)


def variables_lluvia(t):
    t = t.sort_values(["DIVIPOLA", "fecha"]).reset_index(drop=True)
    t["mes"] = t["fecha"].dt.month
    g = t.groupby("DIVIPOLA")["lluvia_mm"]
    t["lluvia_log"] = np.log1p(t["lluvia_mm"])
    t["lluvia_3m_log"] = np.log1p(g.transform(lambda s: s.rolling(3, min_periods=3).sum()))
    clima = (t[t["fecha"] <= CLIMA_FIN].groupby(["DIVIPOLA", "mes"])[["lluvia_log", "lluvia_3m_log"]].mean()
             .rename(columns=lambda c: c + "_clima"))
    t = t.join(clima, on=["DIVIPOLA", "mes"])
    t["lluvia_anom"] = t["lluvia_log"] - t["lluvia_log_clima"]
    t["lluvia_3m_anom"] = t["lluvia_3m_log"] - t["lluvia_3m_log_clima"]
    por_mun = t.groupby("DIVIPOLA")
    t["lluvia_anom_rezago1"] = por_mun["lluvia_anom"].shift(1)
    # Placebo: las mismas variables, pero del mismo mes del año anterior
    t["placebo_anom"] = por_mun["lluvia_anom"].shift(12)
    t["placebo_3m_anom"] = por_mun["lluvia_3m_anom"].shift(12)
    t["placebo_anom_rezago1"] = por_mun["lluvia_anom"].shift(13)
    return t[["DIVIPOLA", "fecha", "lluvia_log", "lluvia_3m_log", "lluvia_anom", "lluvia_3m_anom",
              "lluvia_anom_rezago1", "placebo_anom", "placebo_3m_anom", "placebo_anom_rezago1"]]


USAR_LLUVIA, origen_lluvia = False, "desactivada (INCLUIR_LLUVIA = False)"
if INCLUIR_LLUVIA:
    LLUVIA, origen_lluvia = cargar_lluvia()
    llaves_reto = pd.concat([train, test_pub, test_priv])[["DIVIPOLA", "fecha"]].drop_duplicates()
    USAR_LLUVIA = LLUVIA is not None and validar_lluvia(LLUVIA, llaves_reto)
if USAR_LLUVIA:
    LLUVIA_VARS = variables_lluvia(LLUVIA)
    train, test_pub, test_priv = (d.merge(LLUVIA_VARS, on=["DIVIPOLA", "fecha"], how="left")
                                  for d in (train, test_pub, test_priv))
    resumen_ll = LLUVIA_VARS.assign(anio=LLUVIA_VARS.fecha.dt.year).groupby("anio")["lluvia_anom"].mean()
    print("Anomalía media nacional por año (log; > 0 = más lluvioso de lo normal):", resumen_ll.round(3).to_dict())
print(f"Lluvia: {origen_lluvia} → {'se usa' if USAR_LLUVIA else 'NO se usa'}")

⚠ No se pudo descargar la lluvia: Read failed. See previous exception for details.
Lluvia: no disponible → NO se usa


## 4. Variables y modelo

In [10]:
ONI_SERIE = pd.concat([train, test_pub, test_priv])[["fecha", "ONI"]].groupby("fecha")["ONI"].median()


def features_base(df):
    X = pd.DataFrame(index=df.index)
    for c in COV_ANUALES:
        X["log_" + c] = np.log1p(df[c].clip(lower=0))
    X["ingresos_rango_anio"] = df.groupby("anio")["ingresos_tributarios"].rank(pct=True)
    for c in COV_ESTATICAS + ["ONI"] + HIST_COLS:
        X[c] = df[c]
    for c in [c for c in df.columns if c.endswith("_imputado")]:
        X[c] = df[c]
    for k in (1, 2, 3):
        X[f"ONI_rezago_{k}"] = (df["fecha"] - pd.DateOffset(months=k)).map(ONI_SERIE).to_numpy()
    X["mes"] = df["mes"]
    X["mes_sin"], X["mes_cos"] = np.sin(2 * np.pi * df["mes"] / 12), np.cos(2 * np.pi * df["mes"] / 12)
    X["departamento"] = df["DIVIPOLA"].str[:2].astype(int)
    X["oni_x_inundacion"] = df["ONI"] * df["hist_pct_inund"]
    X["oni_x_tasa"] = df["ONI"] * df["hist_tasa_mun"]
    return X


def features(df, cfg):
    X = features_base(df)
    if cfg.get("espacial"):
        grupos, v = cfg["grupos"], cfg["vecindad"]
        if "ubicacion" in grupos:
            X["lat"], X["lon"] = df["lat"], df["lon"]
        if "poblacion_area" in grupos:
            X["log_area"] = np.log1p(df["area_km2"])
            X["log_poblacion"] = np.log1p(df["densidad_poblacional"].clip(lower=0) * df["area_km2"])
        if "vecinos" in grupos:
            X["hist_vecinos_tasa"], X["hist_vecinos_tasa_mes"] = df[f"vec_{v}_tasa"], df[f"vec_{v}_tasa_mes"]
            if cfg.get("vecinos_plus"):
                X["hist_vecinos_mag"], X["hist_vecinos_reciente"] = df[f"vec_{v}_mag"], df[f"vec_{v}_reciente"]
        if cfg.get("suavizado_espacial"):
            X["hist_tasa_mun_espacial"] = df["hist_tasa_mun_espacial"]
    return X

In [11]:
if "features_sin_lluvia" not in globals():     # evita envolver dos veces si se re-ejecuta la celda
    features_sin_lluvia = features
LLUVIA_ANOM = ["lluvia_anom", "lluvia_3m_anom", "lluvia_anom_rezago1"]


def features(df, cfg):
    X = features_sin_lluvia(df, cfg)
    grupos = cfg.get("lluvia", ())
    if "nivel" in grupos:
        X["lluvia_log"], X["lluvia_3m_log"] = df["lluvia_log"], df["lluvia_3m_log"]
    if "anomalia" in grupos:
        for c in LLUVIA_ANOM:
            X[c] = df[c]
        X["lluvia_anom_x_tasa"] = df["lluvia_anom"] * df["hist_tasa_mes"]
    if "placebo" in grupos:     # mismas columnas, pero con la lluvia de hace 12 meses
        for c in LLUVIA_ANOM:
            X[c] = df[c.replace("lluvia_", "placebo_")]
        X["lluvia_anom_x_tasa"] = df["placebo_anom"] * df["hist_tasa_mes"]
    return X

In [12]:
import shutil
import time

import lightgbm as lgb

NOMBRE_EQUIPO = "AICOLOMBIA"
LGBM_BASE = dict(learning_rate=0.05, num_leaves=31, min_child_samples=100, colsample_bytree=0.8, subsample=0.8,
                 subsample_freq=1, reg_lambda=1.0, deterministic=True, force_row_wise=True, verbose=-1)
ES_MESES, ES_PACIENCIA, FACTOR_ARBOLES, MAX_ARBOLES = 12, 100, 1.15, 3000

SIMPLE = dict(num_leaves=15, min_child_samples=200)
E10S = dict(espacial=True, grupos=("ubicacion", "poblacion_area", "vecinos"), vecindad="k5", vecinos_plus=False,
            suavizado_espacial=False, arboles_p=100, arboles_mag=None, arboles_q=None, lgbm=SIMPLE, semillas=(42,))
E10S_LL = {**E10S, "lluvia": ("nivel", "anomalia")}

CONFIG, NOMBRE_MODELO = E10S_LL, "E10S_LL"
if not USAR_LLUVIA:
    CONFIG, NOMBRE_MODELO = E10S, "E10S"
    print("⚠ Sin lluvia: se usa E10S (LightGBM simple + contexto espacial)")
if not USAR_ESPACIAL:
    CONFIG, NOMBRE_MODELO = {**CONFIG, "espacial": False}, NOMBRE_MODELO + "_sin_espacial"
    print("⚠ Sin cartografía: se omiten las variables espaciales")
print("Modelo:", NOMBRE_MODELO)


def ajustar(X, y, fechas, clase, cfg, n_arboles, semilla, **extra):
    params = {**LGBM_BASE, **cfg["lgbm"], **extra, "random_state": semilla}
    if n_arboles is not None:
        return clase(**params, n_estimators=n_arboles).fit(X, y)
    meses = np.sort(fechas.unique())
    corte = meses[-min(ES_MESES, max(3, len(meses) // 4))]
    a, b = (fechas < corte).to_numpy(), (fechas >= corte).to_numpy()
    m = clase(**params, n_estimators=MAX_ARBOLES).fit(X[a], y[a], eval_set=[(X[b], y[b])],
                                                     callbacks=[lgb.early_stopping(ES_PACIENCIA, verbose=False)])
    n = max(50, int(round((m.best_iteration_ or MAX_ARBOLES) * FACTOR_ARBOLES)))
    return clase(**params, n_estimators=n).fit(X, y)


def entrenar(filas, cfg):
    X, fechas = features(filas, cfg), filas["fecha"]
    y_ev, y_log = filas["tiene_evento"], np.log1p(filas["personas_desplazadas"])
    conjuntos = []
    for s in cfg["semillas"]:
        m = {"p": ajustar(X, y_ev, fechas, lgb.LGBMClassifier, cfg, cfg["arboles_p"], s, objective="binary", metric="auc"),
             "e": ajustar(X, y_log, fechas, lgb.LGBMRegressor, cfg, cfg["arboles_mag"], s, objective="regression", metric="l2")}
        for q in (0.1, 0.9):
            m[q] = ajustar(X, y_log, fechas, lgb.LGBMRegressor, cfg, cfg["arboles_q"], s, objective="quantile",
                           alpha=q, metric="quantile")
        conjuntos.append(m)
    return conjuntos, None


def predecir_log(conjuntos, filas, cfg):
    X = features(filas, cfg)
    p = filas[["DIVIPOLA", "fecha"]].copy()
    p["prob"] = np.mean([m["p"].predict_proba(X)[:, 1] for m in conjuntos], axis=0)
    p["mag"] = np.mean([np.clip(m["e"].predict(X), 0, None) for m in conjuntos], axis=0)
    p["lo"] = np.mean([np.clip(m[0.1].predict(X), 0, None) for m in conjuntos], axis=0)
    p["hi"] = np.mean([np.clip(m[0.9].predict(X), 0, None) for m in conjuntos], axis=0)
    return p


def a_entrega(cruda, factor_q90=1.0):
    p = cruda[["DIVIPOLA", "fecha"]].copy()
    p["prob_evento"] = cruda["prob"].to_numpy()
    p["personas_desplazadas_estimadas"] = np.expm1(cruda["mag"].to_numpy())
    hi = cruda["hi"].to_numpy() * factor_q90
    p["personas_desplazadas_q10"] = np.expm1(np.minimum(cruda["lo"].to_numpy(), hi))
    p["personas_desplazadas_q90"] = np.expm1(np.maximum(cruda["lo"].to_numpy(), hi))
    return p

⚠ Sin lluvia: se usa E10S (LightGBM simple + contexto espacial)
Modelo: E10S


## 5. Validación temporal
Tres cortes que imitan la evaluación: se entrena con la historia hasta una fecha y se predicen meses no vistos, con
las métricas oficiales (T&C 6.3) y la normalización contra la predicción nula. F3 deja un año de distancia, como la
prueba oculta. El factor de `q90` que minimiza el Winkler se elige aquí (mediana de los cortes).

In [13]:
from sklearn.metrics import roc_auc_score

def metricas(pred, verdad):
    m = verdad.merge(pred, on=["DIVIPOLA", "fecha"])
    y = np.log1p(m["personas_desplazadas"])
    est = np.log1p(m["personas_desplazadas_estimadas"])
    lo, hi = np.log1p(m["personas_desplazadas_q10"]), np.log1p(m["personas_desplazadas_q90"])
    wink = (hi - lo) + 10 * np.clip(lo - y, 0, None) + 10 * np.clip(y - hi, 0, None)   # alpha = 0.2
    return {"auc": roc_auc_score(m["tiene_evento"], m["prob_evento"]),
            "rmse_log": float(np.sqrt(np.mean((y - est) ** 2))), "winkler": float(wink.mean())}

def puntos(met, nula):
    """Parte automática de la nota (máx. 75 puntos), como la tabla de posiciones."""
    s_auc = max(0, (met["auc"] - 0.5) / 0.5)
    s_rmse = min(1, max(0, 1 - met["rmse_log"] / nula["rmse_log"]))
    s_wink = min(1, max(0, 1 - met["winkler"] / nula["winkler"]))
    return 100 * (0.50 * s_auc + 0.20 * s_rmse + 0.05 * s_wink)


def prediccion_nula(filas):
    return filas[["DIVIPOLA", "fecha"]].assign(prob_evento=0.5, personas_desplazadas_estimadas=0.0,
                                               personas_desplazadas_q10=0.0, personas_desplazadas_q90=0.0)


def winkler_con_factor(pred, verdad, factor):
    m = verdad.merge(pred, on=["DIVIPOLA", "fecha"])
    y, lo = np.log1p(m["personas_desplazadas"]), np.log1p(m["personas_desplazadas_q10"])
    hi = np.maximum(lo, np.log1p(m["personas_desplazadas_q90"]) * factor)
    return float(((hi - lo) + 10 * np.clip(lo - y, 0, None) + 10 * np.clip(y - hi, 0, None)).mean())

In [14]:
CORTES = [("F1", "2020-09-01", "2020-10-01", "2021-09-01"),   # (corte, fin de la historia, validación desde, hasta)
          ("F2", "2021-09-01", "2021-10-01", "2022-09-01"),
          ("F3", "2020-09-01", "2021-10-01", "2022-09-01")]   # F3: un año de distancia, como la prueba oculta
FACTORES = [0.7, 0.8, 0.9, 1.0, 1.1, 1.25, 1.4]

t0 = time.time()
resultados, factores = {}, []
for corte, fin_hist, ini_val, fin_val in CORTES:
    hist = train[train.fecha <= fin_hist]
    va = train[(train.fecha >= ini_val) & (train.fecha <= fin_val)]
    modelos_cv, _ = entrenar(historial_entrenamiento_completo(hist), CONFIG)
    pred = a_entrega(predecir_log(modelos_cv, historial_completo(hist, va), CONFIG))
    met, nula = metricas(pred, va), metricas(prediccion_nula(va), va)
    winklers = {f: winkler_con_factor(pred, va, f) for f in FACTORES}
    factores.append(min(winklers, key=winklers.get))
    resultados[corte] = {**met, "puntos": puntos(met, nula), "mejor_factor_q90": factores[-1]}

FACTOR_Q90 = float(np.median(factores))
tabla_val = pd.DataFrame(resultados).T
display(tabla_val.round(3))
print(f"{NOMBRE_MODELO} · puntaje de validación (promedio F1–F3): {tabla_val.puntos.mean():.2f} / 75 "
      f"· factor de q90: {FACTOR_Q90} · {time.time() - t0:.0f}s")
print("Referencias del equipo (mismos cortes): E10S_LL 25.56 · E10S 21.98 · V1 20.06")

,auc,rmse_log,winkler,puntos,mejor_factor_q90
F1,0.697,1.352,3.501,21.716,1.1
F2,0.727,1.260,3.295,24.608,0.8
F3,0.680,1.263,3.450,19.618,1.0


E10S · puntaje de validación (promedio F1–F3): 21.98 / 75 · factor de q90: 1.0 · 26s
Referencias del equipo (mismos cortes): E10S_LL 25.56 · E10S 21.98 · V1 20.06


## 6. Entrenar con todo y generar la entrega
Un solo archivo con los dos periodos de prueba (42,978 filas).

In [15]:
PRUEBA = pd.concat([test_pub, test_priv], ignore_index=True)
modelos_fin, _ = entrenar(historial_entrenamiento_completo(train), CONFIG)
entrega = a_entrega(predecir_log(modelos_fin, historial_completo(train, PRUEBA), CONFIG), FACTOR_Q90)
entrega["fecha"] = entrega["fecha"].dt.strftime("%Y-%m-%d")
entrega.head()

,DIVIPOLA,fecha,prob_evento,personas_desplazadas_estimadas,personas_desplazadas_q10,personas_desplazadas_q90
0,05001,2022-10-01,0.610813,4.851811,0.0,137.778121
1,05001,2022-11-01,0.603806,5.631515,0.0,153.274667
2,05001,2022-12-01,0.384695,2.108280,0.0,54.175778
3,05001,2023-01-01,0.264728,1.723832,0.0,31.652359
4,05001,2023-02-01,0.379498,1.408993,0.0,21.706311


In [16]:
clasificador = modelos_fin[0]["p"]
imp = pd.Series(clasificador.booster_.feature_importance("gain"), index=clasificador.feature_name_)
imp = imp / imp.sum()
print(f"Importancia de la lluvia en el clasificador: {imp[imp.index.str.startswith('lluvia')].sum():.1%}")
display(imp.sort_values(ascending=False).head(15).round(3).to_frame("importancia (ganancia)"))

Importancia de la lluvia en el clasificador: 0.0%


,importancia (ganancia)
hist_tasa_mes,0.137
hist_esperado_log,0.088
oni_x_tasa,0.077
log_poblacion,0.073
hist_tasa_dep_mes,0.069
hist_tasa_mun,0.063
hist_reciente_12,0.046
departamento,0.042
hist_vecinos_tasa_mes,0.036
ONI,0.028


### Revisar el formato antes de enviar
Esta función aplica las mismas reglas que el calificador. Si dice ✅, su archivo será aceptado.

In [17]:
COLUMNAS = ["DIVIPOLA", "fecha", "prob_evento", "personas_desplazadas_estimadas",
            "personas_desplazadas_q10", "personas_desplazadas_q90"]

def validar_entrega(e, test_pub=test_pub, test_priv=test_priv):
    errores = []
    faltan = [c for c in COLUMNAS if c not in e.columns]
    if faltan:
        return print("❌ Faltan columnas:", faltan)
    claves = pd.concat([test_pub, test_priv])[["DIVIPOLA", "fecha"]].assign(fecha=lambda d: pd.to_datetime(d.fecha))
    e2 = e.assign(DIVIPOLA=e.DIVIPOLA.astype(str).str.zfill(5), fecha=pd.to_datetime(e.fecha))
    if e2.duplicated(["DIVIPOLA", "fecha"]).any():
        errores.append("hay filas DIVIPOLA-fecha duplicadas")
    faltantes = len(claves.merge(e2, on=["DIVIPOLA", "fecha"], how="left", indicator=True).query("_merge == 'left_only'"))
    if faltantes:
        errores.append(f"faltan {faltantes} filas de las {len(claves)} esperadas")
    num = e2[COLUMNAS[2:]].apply(pd.to_numeric, errors="coerce")
    if num.isna().any().any():
        errores.append("hay valores vacíos o no numéricos")
    if not num.prob_evento.between(0, 1).all():
        errores.append("prob_evento debe estar entre 0 y 1")
    if (num[COLUMNAS[3:]] < 0).any().any():
        errores.append("las columnas de personas deben ser >= 0")
    if (num.personas_desplazadas_q10 > num.personas_desplazadas_q90).any():
        errores.append("personas_desplazadas_q10 debe ser <= personas_desplazadas_q90")
    print("✅ Entrega válida:", len(e), "filas" if not errores else "", *["\n❌ " + x for x in errores])
    return not errores

validar_entrega(entrega)

✅ Entrega válida: 42978 filas


True

### Guardar y descargar
Suban este archivo al formulario de entrega. Máximo 1 entrega cada 30 minutos; cuenta la última válida antes del
cierre.

In [18]:
NOMBRE_EQUIPO = "AICOLOMBIA"
archivo = f"entrega_{NOMBRE_EQUIPO}.csv"
entrega.to_csv(archivo, index=False)
try:
    from google.colab import files
    files.download(archivo)
except ImportError:
    print("Guardado en", os.path.abspath(archivo))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Otros entregables (T&C, Artículo 5)

### Tabla de replicabilidad
| Variable usada | Fuente en este dataset / externa | Proxy abierto y global | Fuente del proxy |
|---|---|---|---|
| Historial de eventos (tasas por municipio, mes, departamento; magnitud típica; % inundaciones y movimientos en masa; tasa reciente) | UNGRD (dataset del reto) | Registros de desastres y desplazamiento | DesInventar Sendai (UNDRR), EM-DAT, IDMC |
| `densidad_poblacional` y población (densidad × área) | TerriData (DNP) + área de la cartografía | Población | WorldPop, GHS-POP |
| `ingresos_tributarios` | TerriData (DNP) | Actividad económica subnacional | Luces nocturnas VIIRS; PIB subnacional grillado (Kummu et al.) |
| `inversion_gestion_riesgo` | TerriData (DNP) | Capacidad de afrontamiento | INFORM Subnational |
| `nbi_pct` | TerriData / Censo 2018 | Pobreza multidimensional | Relative Wealth Index (Meta), DHS, MPI subnacional (OPHI) |
| `elevacion_media`, `pendiente_media`, `pct_pendiente_alta` | Copernicus DEM | Elevación y pendiente | Copernicus DEM GLO-90 / SRTM |
| `ONI` y rezagos | NOAA CPC | ONI | NOAA CPC |
| Mes del año | — | Calendario | Universal |
| Latitud, longitud, área, vecinos | **Externa:** UPRA, capa municipios_generalizada (MGN del DANE) | Límites administrativos | geoBoundaries ADM2, GADM |
| Lluvia: nivel, acumulado de 3 meses, anomalías | **Externa:** CHIRPS v2.0 mensual, 2018-01 → 2025-12 | Lluvia | CHIRPS (ya es global, 50° S – 50° N); ERA5-Land |

### Geovisor
Mapa interactivo con la probabilidad, el estimado de personas y el intervalo por municipio y mes (entregado aparte).

### Uso de IA generativa (T&C 9.1)
| Campo | Detalle |
|---|---|
| Herramienta | Claude Code (Anthropic), aplicación de escritorio |
| Modelo / versión | Claude Opus 5.5 (`claude-opus-5-5`) |
| Alcance | Lectura de los T&C y del diccionario; propuesta de alternativas de modelado; escritura y depuración del código (tratamiento, historial, cartografía, lluvia, modelos, métricas, validación); notebooks de experimentos (rondas 1–5); documentación y tabla de replicabilidad |
| Dirección humana | El integrante decidió qué experimentos correr, ejecutó los notebooks, verificó los resultados y eligió la configuración final (E10S_LL) con la validación temporal y una regla fijada de antemano (mejorar ≥ 0.5 puntos sin empeorar el corte F3). La IA fue apoyo, no autora de las decisiones |

### Créditos
Datos del reto (UNGRD, TerriData/DNP, Copernicus DEM, NOAA CPC), entregados por el PNUD y no redistribuidos ·
Cartografía: UPRA (MGN del DANE) · Lluvia: CHIRPS v2.0 (Funk et al., 2015, *Scientific Data* 2:150066) ·
Librerías: pandas, NumPy, scikit-learn (BSD-3), LightGBM (MIT), Shapely (BSD-3), rasterio (BSD-3), gdown (MIT).